In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Pearson Feature Correlation Heatmap (`plots/pearson_correlation.ipynb`)

This notebook computes the pairwise **Pearson Correlation Matrix** across all predictor features configured via `config/triage_conf.json` (`age`, `gender`, `cc_breathingdifficulty` + 16 Continuous Vital Delta & Range Features):

### Workflow & Outputs
1. **Config & Feature Ingestion**: Loads data source specified in `triage_conf.json` and extracts the 19 clinical predictor features.
2. **Pearson Correlation Calculation**: Computes exact pairwise linear correlation coefficients $r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$ across all feature pairs.
3. **Heatmap Visualization**: Generates a high-resolution, color-coded correlation heatmap with numeric overlays saved to `plots/pearson_correlation_heatmap.png`.
4. **CSV Export**: Writes the full symmetric $19 \times 19$ matrix report to `reports/pearson_correlation_matrix.csv`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(reshape2)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Build 19 Predictor Feature Matrix
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
df_features <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
df_features <- na.omit(df_features)
cat(sprintf("Complete cases ready for correlation analysis: %d rows x %d features\n", nrow(df_features), ncol(df_features)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Compute Pearson Correlation Matrix & Export CSV Report
# ---------------------------------------------------------
cor_matrix <- cor(df_features, method = "pearson", use = "pairwise.complete.obs")
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
csv_path <- file.path(reports_dir, "pearson_correlation_matrix.csv")
write.csv(cor_matrix, file = csv_path, row.names = TRUE)
cat("Full Pearson Correlation Matrix successfully written to CSV:", csv_path, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Render & Save Publication-Quality Correlation Heatmap
# ---------------------------------------------------------
cor_melted <- melt(cor_matrix)
colnames(cor_melted) <- c("Feature1", "Feature2", "Correlation")
p_heat <- ggplot(cor_melted, aes(x = Feature1, y = Feature2, fill = Correlation)) +
  geom_tile(color = "white", linewidth = 0.3) +
  scale_fill_gradient2(low = "#2b5c8f", mid = "#ffffff", high = "#e07a5f", midpoint = 0, limit = c(-1, 1), name = "Pearson\nCorrelation") +
  geom_text(aes(label = sprintf("%.2f", Correlation)), color = "black", size = 2.5) +
  theme_minimal() +
  labs(title = "Pearson Correlation Matrix Heatmap",
       subtitle = "Feature Correlation Analysis for Emergency Triage Predictors",
       x = "", y = "") +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, size = 9, face = "bold"),
    axis.text.y = element_text(size = 9, face = "bold"),
    plot.title = element_text(face = "bold", size = 14, hjust = 0.5),
    plot.subtitle = element_text(size = 11, hjust = 0.5),
    legend.position = "right"
  )
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
plot_file <- file.path(plots_dir, "pearson_correlation_heatmap.png")
ggsave(plot_file, plot = p_heat, width = 12, height = 10, dpi = 300)
cat("Pearson Correlation Heatmap saved to:", plot_file, "\n")
p_heat